# Clase 14 — Transformers, LLMs y RAG (Versión Avanzada)
**Autor:** Josef Rodriguez

## Objetivos
- Explicar la intuición de transformers
- Visualizar self-attention
- Usar sentence embeddings modernos
- Crear gráficos 2D, 3D y t-SNE
- Implementar búsqueda semántica
- Construir una mini demo de chatbot RAG
- Dejar una demo opcional con LangChain + Ollama


## Instalación sugerida
```bash
pip install sentence-transformers plotly scikit-learn pandas matplotlib
pip install -U langchain langchain-ollama
```


## Evolución del NLP

BoW → TF-IDF → Word2Vec → Transformers → LLMs → RAG

Los transformers introducen embeddings **contextuales**, es decir, el vector depende de la oración donde aparece la palabra.


## Intuición de Self-Attention

Cada token evalúa a qué otros tokens debe prestar atención.

La idea matemática general es:

Attention(Q, K, V) = softmax(QKᵀ / √d) V


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer


## Matriz de atención conceptual


In [ ]:
tokens = ["The", "animal", "was", "too", "tired"]
np.random.seed(42)

A = np.random.rand(len(tokens), len(tokens))
A = A / A.sum(axis=1, keepdims=True)

plt.figure(figsize=(7, 5))
plt.imshow(A)
plt.colorbar()
plt.xticks(range(len(tokens)), tokens)
plt.yticks(range(len(tokens)), tokens)
plt.title("Matriz de Self-Attention (conceptual)")
plt.xlabel("Token atendido")
plt.ylabel("Token que atiende")
plt.show()


## Sentence Embeddings modernos
Usaremos un modelo liviano y popular.


In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")


In [ ]:
sentences = [
    "Machine learning is amazing",
    "Artificial intelligence is transforming the world",
    "Spam messages are annoying",
    "This email is probably spam",
    "Transformers use self attention",
    "RAG combines retrieval and generation",
    "Deep learning models can be very powerful",
    "Natural language processing works with text",
    "The cat is sleeping on the sofa",
    "Semantic search retrieves relevant documents"
]

emb = model.encode(sentences)
emb.shape


## Matriz de similitud entre oraciones


In [ ]:
sim = cosine_similarity(emb)

sim_df = pd.DataFrame(sim, index=[f"S{i+1}" for i in range(len(sentences))],
                      columns=[f"S{i+1}" for i in range(len(sentences))])
sim_df.round(3)


In [ ]:
plt.figure(figsize=(9, 7))
plt.imshow(sim)
plt.colorbar()
plt.xticks(range(len(sentences)), [f"S{i+1}" for i in range(len(sentences))], rotation=45)
plt.yticks(range(len(sentences)), [f"S{i+1}" for i in range(len(sentences))])
plt.title("Heatmap de similitud coseno")
plt.show()


## Visualización 2D con PCA


In [ ]:
pca2 = PCA(n_components=2)
coords2 = pca2.fit_transform(emb)

plt.figure(figsize=(10, 7))
plt.scatter(coords2[:, 0], coords2[:, 1])

for i, s in enumerate(sentences):
    plt.text(coords2[i, 0], coords2[i, 1], f"S{i+1}", fontsize=9)

plt.title("Sentence Embeddings en 2D")
plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.show()


## Visualización 3D interactiva con Plotly


In [ ]:
pca3 = PCA(n_components=3)
coords3 = pca3.fit_transform(emb)

plot_df = pd.DataFrame({
    "sentence": [f"S{i+1}" for i in range(len(sentences))],
    "text": sentences,
    "x": coords3[:, 0],
    "y": coords3[:, 1],
    "z": coords3[:, 2],
})

fig = px.scatter_3d(
    plot_df,
    x="x",
    y="y",
    z="z",
    text="sentence",
    hover_data=["text"],
    title="Sentence Embeddings en 3D"
)
fig.update_traces(marker=dict(size=6))
fig.show()


## Visualización t-SNE profesional


In [ ]:
tsne = TSNE(n_components=2, perplexity=5, random_state=42, init="pca", learning_rate="auto")
coords_tsne = tsne.fit_transform(emb)

plt.figure(figsize=(10, 7))
plt.scatter(coords_tsne[:, 0], coords_tsne[:, 1])

for i, s in enumerate(sentences):
    plt.text(coords_tsne[i, 0], coords_tsne[i, 1], f"S{i+1}", fontsize=9)

plt.title("Sentence Embeddings con t-SNE")
plt.show()


## ¿Qué es RAG?

RAG = Retrieval Augmented Generation

Flujo:
1. convertir documentos a embeddings
2. convertir la consulta a embedding
3. buscar los documentos más cercanos
4. pasar ese contexto al LLM
5. generar una respuesta grounded en documentos


In [ ]:
docs = [
    "Machine learning is a field of artificial intelligence focused on learning from data.",
    "Spam detection is a classic NLP classification problem.",
    "Transformers use self attention to model relationships between tokens.",
    "RAG combines retrieval and generation to answer questions with external knowledge.",
    "Clustering groups similar observations without using labels.",
    "TF-IDF weights terms according to their importance in the corpus.",
    "Sentence embeddings enable semantic search over documents.",
    "Ollama lets you run open source language models locally."
]

doc_emb = model.encode(docs)
doc_emb.shape


## Búsqueda semántica simple


In [ ]:
query = "How do transformers understand context?"
q_emb = model.encode([query])

scores = cosine_similarity(q_emb, doc_emb)[0]
ranking = pd.DataFrame({"document": docs, "score": scores}).sort_values("score", ascending=False)
ranking


In [ ]:
top_k = 3
top_docs = ranking.head(top_k)["document"].tolist()

for i, d in enumerate(top_docs, 1):
    print(f"Top {i}: {d}\n")


## Mini chatbot RAG sin framework


In [ ]:
def mini_rag_answer(question, documents, embed_model, top_k=2):
    D = embed_model.encode(documents)
    q = embed_model.encode([question])
    scores = cosine_similarity(q, D)[0]
    idx = np.argsort(scores)[::-1][:top_k]
    context = "\n".join([f"- {documents[i]}" for i in idx])

    answer = f"""Pregunta: {question}

Contexto recuperado:
{context}

Respuesta simulada:
Con base en el contexto recuperado, la respuesta debe apoyarse principalmente en esos fragmentos.
"""
    return answer

print(mini_rag_answer(
    "What is RAG and why is it useful?",
    docs,
    model,
    top_k=2
))


## Demo opcional con LangChain + Ollama

Esta parte es opcional y requiere:
- tener Ollama instalado y corriendo localmente
- haber descargado un modelo con `ollama pull`
- instalar `langchain-ollama`

El ejemplo siguiente muestra una versión mínima de RAG local.


In [ ]:
# OPCIONAL: descomentar solo si tienes Ollama instalado localmente

# from langchain_ollama import ChatOllama, OllamaEmbeddings
# from langchain_core.vectorstores import InMemoryVectorStore
# from langchain_core.documents import Document

# llm = ChatOllama(model="llama3.2")
# emb_model = OllamaEmbeddings(model="nomic-embed-text")

# lc_docs = [Document(page_content=d) for d in docs]
# vector_store = InMemoryVectorStore(emb_model)
# vector_store.add_documents(lc_docs)

# retriever = vector_store.as_retriever(search_kwargs={"k": 2})

# question = "Explain RAG in simple terms"
# retrieved = retriever.invoke(question)

# context = "\n".join([doc.page_content for doc in retrieved])

# prompt = f"""Answer the question using only the context below.

# Context:
# {context}

# Question:
# {question}
# """

# response = llm.invoke(prompt)
# print(response.content)


## Mini mapa mental final

Transformers  
↓  
embeddings contextuales  
↓  
búsqueda semántica  
↓  
RAG  
↓  
chatbots con documentos

Esta es una de las arquitecturas más usadas hoy en productos de IA aplicada.
